# 🛡️ CyberRott v1.2 — Pipeline de Treinamento e Alinhamento por Reforço Online (GRPO)

Arquitetura unificada de treinamento para o **CyberRott v1.1 / v1.2 (Phi-4-mini 3.8B)** implementando o motor de **Group Relative Policy Optimization (GRPO)** — a mesma formulação matemática de RL do **DeepSeek-R1** — nativamente com Hugging Face `TRL` e `PEFT`.

```text
===================================================================================================
                       FLUXO DE TREINAMENTO E AUTO-EVOLUÇÃO (CYBERROTT v1.2)
===================================================================================================

+---------------------------------------------------------------------------------------------------+
| FASE 1: Dataset Consolidado (~4.68k Amostras CoT Open-Source em ChatML)                          |
| - Amostras ricas em verificação técnica (CWE, MITRE ATT&CK, Múltipla Escolha e Código)            |
| - 16x Golden Anchors Ponderadas + Scrubber de Recusas & CoT de 7 Estágios (<think>)               |
| - Carregamento direto via GitHub (data/cyberrott_final_train.jsonl) sem retrabalho                |
+---------------------------------------------------------------------------------------------------+
                                                  │
                                                  ▼
+---------------------------------------------------------------------------------------------------+
| FASE 2: Treinamento SFT Base LoRA v1.1 (Detecção Inteligente de Checkpoint no Drive)              |
| - Se o checkpoint SFT já existir no Drive/local, pula o retreino e carrega instantaneamente!     |
| - Caso contrário: LoRA r=32, all-linear, Batch Efetivo=16, BF16/TF32 na GPU L4/A100               |
+---------------------------------------------------------------------------------------------------+
                                                  │
                                                  ▼
+---------------------------------------------------------------------------------------------------+
| FASE 3: Treinamento Online GRPO v1.2 (Group Relative Policy Optimization - DeepSeek-R1 Style)      |
| 1. In-Memory SFT Merge: Funde o LoRA SFT na memória em segundos -> π_ref com ZERO VRAM extra!    |
| 2. Geração em Grupo: G=4 rollouts exploratórios por prompt calculando vantagem relativa intra-grupo |
| 3. Suíte de 3 Recompensas Determinísticas Verificáveis (STaRVerifier):                            |
|    • cot_integrity_reward (0.30): <think> tags, 7 estágios analíticos, anti-recusa estrita        |
|    • technical_accuracy_reward (0.50): Gabarito exato de múltipla escolha e CWE/MITRE matches      |
|    • sandbox_code_reward (0.20): Compilação AST Python, gramática YARA e regras Sigma/YAML        |
| 4. Otimização Sem Crítico (Critic-Free): Economiza ~15GB de VRAM, viabilizando A100 e L4          |
+---------------------------------------------------------------------------------------------------+
                                                  │
                                                  ▼
+---------------------------------------------------------------------------------------------------+
| FASE 4: Auditoria de Vulnerabilidades & Exportação Final (Tripartite Output)                      |
| - Avaliação com Raciocínio CoT (<think>) + Diagnóstico Técnico (FATO) + Correção em Sandbox       |
| - Sincronização automática do checkpoint GRPO final para o Google Drive                           |
+---------------------------------------------------------------------------------------------------+
```

In [ ]:
# 1. Instalação das dependências modernas (compatibilidade estrita com PyTorch 2.x e CUDA 12)
!pip install -q "transformers>=4.46.0,<5.0.0" "peft>=0.13.0,<0.16.0" "trl>=0.12.0,<0.17.0" "accelerate>=1.0.0" "datasets>=3.0.0" "bitsandbytes>=0.44.0" "pyyaml" "vllm"

# 2. Diagnóstico de Aceleração de Hardware GPU (L4 / A100)
import os
import torch

print("=" * 70)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    major, minor = torch.cuda.get_device_capability(0)
    bf16_supported = torch.cuda.is_bf16_supported()

    # Ativação de TF32 para arquiteturas modernas (Ampere, Ada Lovelace, Hopper)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    print(f"Dispositivo GPU: {device_name}")
    print(f"VRAM Total:      {total_vram_gb:.1f} GB")
    print(f"Arquitetura:     Compute Capability {major}.{minor} (Ampere/Ada: {major >= 8})")
    print(f"Suporte BF16:    {bf16_supported}")
    print(f"TF32 TensorCore: {'Ativado com sucesso' if major >= 8 else 'Não disponível'}")
else:
    print("GPU não detectada! Ative GPU em: Runtime -> Change runtime type -> Hardware accelerator.")
print("=" * 70)
!nvidia-smi


In [ ]:
import os
import sys

# 1. Montagem Opcional do Google Drive (Persistência Segura de Checkpoints)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DRIVE_DIR = "/content/drive/MyDrive/CyberRott_Checkpoints"
    os.makedirs(CHECKPOINT_DRIVE_DIR, exist_ok=True)
    print(f"[✓] Google Drive montado com sucesso em: {CHECKPOINT_DRIVE_DIR}")
except Exception:
    print("[*] Google Drive não montado (ambiente local ou execução independente).")

# 2. Localização e Sincronização do Repositório CyberRott1.0
REPO_DIR = "/content/CyberRott1.0"
if os.path.exists("/content"):
    if not os.path.exists(REPO_DIR):
        print("[*] Clonando repositório CyberRott1.0 no ambiente Colab...")
        !git clone https://github.com/johnknevescarrijo/CyberRott.git {REPO_DIR}
    else:
        print("[*] Repositório CyberRott1.0 já existente. Atualizando via git pull...")
        !git -C {REPO_DIR} pull origin main
    os.chdir(REPO_DIR)
elif os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull origin main
    os.chdir(REPO_DIR)

print(f"[✓] Diretório de Trabalho: {os.getcwd()}")
sys.path.insert(0, os.getcwd())

# 3. Configuração Segura do HF_TOKEN (Google Colab Secrets ou Variável de Ambiente)
try:
    from google.colab import userdata
    hf_secret = userdata.get("HF_TOKEN")
    if hf_secret:
        os.environ["HF_TOKEN"] = hf_secret
        print("[✓] HF_TOKEN carregado com sucesso do Google Colab Secrets.")
except Exception:
    pass

if "HF_TOKEN" in os.environ and os.environ["HF_TOKEN"]:
    print("[✓] HF_TOKEN configurado no ambiente.")
else:
    print("[*] HF_TOKEN não detectado nos Secrets (opcional para modelos públicos).")


---
## 📦 FASE 1: Dataset Consolidado v1.1 / v1.2 (`data/cyberrott_final_train.jsonl`)

O repositório já inclui por padrão o corpus consolidado `data/cyberrott_final_train.jsonl` (4.680 amostras / ~41 MB) com raciocínio analítico profundo de 7 estágios, metadados de verificação (CWE, MITRE ATT&CK, gabaritos de múltipla escolha e blocos de código) e as Âncoras de Ouro em `data/dataset_cyber_ready.jsonl`.

Se o arquivo já estiver presente no diretório clonado do GitHub, **ele é utilizado diretamente sem necessidade de download ou regeneração**.

In [ ]:
import os
import json
import urllib.request

DATASET_FILE = "data/cyberrott_final_train.jsonl"
ANCHOR_FILE = "data/dataset_cyber_ready.jsonl"

def count_samples(filepath):
    if not os.path.exists(filepath):
        return 0
    with open(filepath, "r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

# Checagem: se o dataset v1.1 completo já existir com >= 4500 amostras, usa diretamente
existing_samples = count_samples(DATASET_FILE)
if existing_samples >= 4500:
    print(f"[✓] Dataset completo ({existing_samples:,} amostras / {os.path.getsize(DATASET_FILE)/(1024*1024):.2f} MB) já presente localmente via Git! Pulando geração...")
else:
    print(f"[*] Dataset local ausente ou incompleto ({existing_samples} amostras). Baixando versão consolidada do GitHub...")
    os.makedirs("data", exist_ok=True)
    raw_url = "https://raw.githubusercontent.com/johnknevescarrijo/CyberRott1.0/main/data/cyberrott_final_train.jsonl"
    try:
        urllib.request.urlretrieve(raw_url, DATASET_FILE)
        print(f"[✓] Download do dataset final concluído: {count_samples(DATASET_FILE):,} amostras.")
    except Exception as e:
        print(f"[!] Erro no download direto ({e}). Executando pipeline data_prep.py...")
        !python scripts/data_prep.py \
            --preset v1.1 \
            --anchor-file data/dataset_cyber_ready.jsonl \
            --output-file data/cyberrott_final_train.jsonl \
            --seed 42

# Validação e Estatísticas do Corpus
if os.path.exists(DATASET_FILE):
    size_mb = os.path.getsize(DATASET_FILE) / (1024 * 1024)
    total_samples = count_samples(DATASET_FILE)
    print(f"\n[✓] Corpus de treino pronto: {total_samples:,} amostras ({size_mb:.2f} MB)")

    # Visualização de amostra de auditoria
    with open(DATASET_FILE, "r", encoding="utf-8") as f:
        sample = json.loads(f.readline())
    print("\n" + "=" * 60)
    print("EXEMPLO DE AMOSTRA CURADA (ChatML):")
    print("=" * 60)
    for msg in sample.get("messages", []):
        snippet = msg["content"][:250] + ("..." if len(msg["content"]) > 250 else "")
        print(f"[{msg["role"].upper()}]:\n{snippet}\n")
else:
    print(f"[Erro] Arquivo {DATASET_FILE} não foi localizado.")

---
## 🚀 FASE 2: Checkpoint SFT Base LoRA (`cyberrott_v1_1_sft_lora`)

O modelo GRPO (Fase 3) necessita de um checkpoint supervisionado (`SFT_LORA`) para inicializar a política e servir de referência ($\pi_{\text{ref}}$).

### Verificação Inteligente de Checkpoint:
- Se você já treinou ou sincronizou o checkpoint do `SFT_LORA` no **Google Drive** (em `/content/drive/MyDrive/CyberRott_Checkpoints/cyberrott_v1_1_sft_lora` ou `/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott_v1_1_Final`) ou no disco local (`./cyberrott_v1_1_sft_lora`), ele será **detectado e reutilizado automaticamente, pulando o retreino SFT**!
- Se nenhum checkpoint for encontrado, o script `scripts/train_colab.py` é executado para treinar o SFT LoRA em `bfloat16`, salvando automaticamente uma cópia de segurança no Google Drive.

In [ ]:
import os
import shutil

# Verificação Inteligente de Checkpoint SFT LoRA
LOCAL_SFT_DIR = "./cyberrott_v1_1_sft_lora"
CANDIDATE_PATHS = [
    LOCAL_SFT_DIR,
    "/content/drive/MyDrive/CyberRott_Checkpoints/cyberrott_v1_1_sft_lora",
    "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott_v1_1_Final",
]

sft_checkpoint_found = None
for path in CANDIDATE_PATHS:
    if os.path.exists(path) and os.path.exists(os.path.join(path, "adapter_config.json")):
        sft_checkpoint_found = path
        break

if sft_checkpoint_found:
    print(f"[✓] Checkpoint SFT LoRA prévio encontrado em: '{sft_checkpoint_found}'!")
    if sft_checkpoint_found != LOCAL_SFT_DIR:
        print(f"[*] Sincronizando adaptador para o diretório local '{LOCAL_SFT_DIR}'...")
        shutil.copytree(sft_checkpoint_found, LOCAL_SFT_DIR, dirs_exist_ok=True)
    print("[✓] Checkpoint SFT LoRA pronto para fusão na Fase 3. RETREINO SFT PULADO COM SUCESSO!")
else:
    print("[*] Nenhum checkpoint SFT LoRA prévio encontrado. Iniciando treinamento SFT da Fase 2...")
    !python scripts/train_colab.py \
        --data_path data/cyberrott_final_train.jsonl \
        --output_dir ./cyberrott_v1_1_sft_lora \
        --epochs 3 \
        --per_device_train_batch_size 2 \
        --gradient_accumulation_steps 8 \
        --max_seq_length 2048 \
        --lora_r 32 \
        --lora_alpha 64 \
        --torch_dtype bfloat16 \
        --attn_implementation sdpa \
        --learning_rate 2e-4

    # Backup automático no Google Drive
    if os.path.exists("/content/drive/MyDrive/CyberRott_Checkpoints") and os.path.exists(LOCAL_SFT_DIR):
        drive_dest = "/content/drive/MyDrive/CyberRott_Checkpoints/cyberrott_v1_1_sft_lora"
        print(f"[*] Salvando backup do SFT LoRA no Google Drive: {drive_dest}...")
        shutil.copytree(LOCAL_SFT_DIR, drive_dest, dirs_exist_ok=True)
        print("[✓] Backup do SFT LoRA concluído com sucesso!")

---
## 🧠 FASE 3: Treinamento por Reforço Online via GRPO (DeepSeek-R1 Style)

Substituindo o antigo ciclo offline (`scripts/train_star.py`), o **CyberRott 1.2** adota **Group Relative Policy Optimization (GRPO)** — a mesma formulação matemática do DeepSeek-R1 — utilizando Hugging Face `TRL` (`GRPOTrainer` e `GRPOConfig`):

### Vantagens do GRPO no CyberRott:
1. **Fusão em Memória Instantânea ($\pi_{\text{ref}}$ com Zero VRAM Extra):**
   O checkpoint `cyberrott_v1_1_sft_lora` da Fase 2 é fundido no modelo base (`microsoft/Phi-4-mini-instruct`) na memória via `merge_and_unload()` em poucos segundos. Ao desativar temporariamente o novo adaptador LoRA durante o cálculo de logprobs, o modelo fundido atua como a política de referência $\pi_{\text{ref}}$ **sem precisar carregar um segundo modelo de 3.8B na VRAM**!
2. **Arquitetura Sem Crítico (Critic-Free):**
   Elimina completamente a rede de valor ($V_\phi$), reduzindo drasticamente o consumo de VRAM e permitindo treinar com estabilidade total em uma GPU única (NVIDIA A100 40GB ou L4 24GB).
3. **Vantagem Relativa em Grupo ($G=4$):**
   Gera 4 rollouts exploratórios por prompt e estima a vantagem normalizada intra-grupo: $A_i = \frac{R_i - \text{mean}(R)}{\text{std}(R) + \epsilon}$.
4. **Suíte de 3 Funções de Recompensa Determinísticas Verificáveis (STaRVerifier):**
   - **`cot_integrity_reward` (peso 0.30):** Integridade de `<think>...</think>`, 7 estágios analíticos e penalidade imediata (0.0) para recusas de IA.
   - **`technical_accuracy_reward` (peso 0.50):** Gabarito exato de múltipla escolha (A, B, C, D) e presença de CWE IDs / MITRE ATT&CK.
   - **`sandbox_code_reward` (peso 0.20):** Compilação sintática AST de código Python, regras YARA e regras Sigma/YAML.

*(Nota: O pipeline anterior de Rejection Sampling offline continua acessível via `scripts/train_star.py` para fins de benchmark histórico).*

In [ ]:
# Executa o Treinamento Online GRPO (CyberRott v1.2)
!python scripts/train_grpo.py \
    --model_name_or_path microsoft/Phi-4-mini-instruct \
    --adapter_name_or_path ./cyberrott_v1_1_sft_lora \
    --dataset_name data/cyberrott_final_train.jsonl \
    --anchor_file data/dataset_cyber_ready.jsonl \
    --output_dir ./CyberRott-v1.2-GRPO \
    --num_generations 4 \
    --max_prompt_length 768 \
    --max_completion_length 512 \
    --max_samples 80 \
    --learning_rate 5e-6 \
    --beta 0.04 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 1 \
    --save_steps 20 \
    --num_train_epochs 1 \
    --lora_r 32 \
    --lora_alpha 64 \
    --torch_dtype bfloat16 \
    --attn_implementation sdpa \
    --use_vllm


---
## 🎯 FASE 4: Inferência, Auditoria e Backup do Checkpoint Final GRPO

Nesta fase final:
1. Carregamos o modelo otimizado pelo pipeline GRPO (`./CyberRott-v1.2-GRPO`) com fallback seguro para o SFT LoRA.
2. Executamos um teste de auditoria técnica de cibersegurança em código C para verificar o raciocínio profundo de 7 estágios dentro de `<think>...</think>` e o formato conclusivo tripartite (`FATO:`, `DETECÇÃO:`, `SUGESTÃO DE CORREÇÃO:`).
3. Sincronizamos o checkpoint final com o Google Drive para persistência definitiva.

In [ ]:
import os
import shutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Identificar o Checkpoint Final (Prioridade: GRPO -> SFT)
FINAL_CHECKPOINT = "./CyberRott-v1.2-GRPO"
if not (os.path.exists(FINAL_CHECKPOINT) and os.path.exists(os.path.join(FINAL_CHECKPOINT, "adapter_config.json"))):
    for candidate in ["./cyberrott_v1_1_sft_lora", "/content/drive/MyDrive/CyberRott_Checkpoints/cyberrott_v1_1_sft_lora"]:
        if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, "adapter_config.json")):
            FINAL_CHECKPOINT = candidate
            break

print(f"[*] Checkpoint Selecionado: {FINAL_CHECKPOINT}")

BASE_MODEL = "microsoft/Phi-4-mini-instruct"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32

# Carrega Tokenizer e Modelo Base em bfloat16
tokenizer = AutoTokenizer.from_pretrained(
    FINAL_CHECKPOINT if os.path.exists(os.path.join(FINAL_CHECKPOINT, "tokenizer_config.json")) else BASE_MODEL,
    trust_remote_code=False
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else "cpu",
    trust_remote_code=False,
)

# Mescla pesos LoRA com o modelo base para inferência direta
print(f"[*] Mesclando pesos adaptadores de {FINAL_CHECKPOINT}...")
model = PeftModel.from_pretrained(base_model, FINAL_CHECKPOINT).merge_and_unload()
model.eval()
print("[✓] Modelo CyberRott v1.2 GRPO pronto para auditoria técnica!")

# 2. Desafio de Auditoria em C (Buffer Overflow / Taint Analysis)
audit_prompt = (
    "Realize a auditoria de segurança rigorosa da seguinte função C em espaço de kernel/usuário:\n\n"
    "```c\n"
    "void parse_network_packet(const char *incoming_payload, int packet_len) {\n"
    "    char local_buffer[64];\n"
    "    if (packet_len > 0) {\n"
    "        strcpy(local_buffer, incoming_payload);\n"
    "    }\n"
    "    process_packet_header(local_buffer);\n"
    "}\n"
    "```\n\n"
    "Instruções:\n"
    "1. Desenvolva seu raciocínio analítico dentro da tag <think>...</think>.\n"
    "2. Declare a vulnerabilidade identificada, o CWE correspondente e o impacto.\n"
    "3. Forneça o patch corretivo defensivo com verificação rígida de limites."
)

messages = [
    {
        "role": "system",
        "content": "Você é o CyberRott, uma IA especialista em segurança ofensiva, defensiva e análise de vulnerabilidades com raciocínio analítico profundo."
    },
    {"role": "user", "content": audit_prompt}
]

formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

print("\n" + "=" * 70)
print("🛡️ RESPOSTA DO CYBERROTT (RACIOCÍNIO TRIPARTITE):")
print("=" * 70 + "\n")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

input_len = inputs["input_ids"].shape[1]
output_text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print(output_text)

# 3. Backup para o Google Drive
DRIVE_BACKUP = "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott_v1_2_GRPO"
if os.path.exists("/content/drive/MyDrive"):
    print(f"\n[*] Sincronizando checkpoint final para o Google Drive: {DRIVE_BACKUP}...")
    shutil.copytree(FINAL_CHECKPOINT, DRIVE_BACKUP, dirs_exist_ok=True)
    print("[✓] Backup no Google Drive concluído com sucesso!")

---
## 🏆 FASE 5: Avaliação Formal de Benchmark WMDP-Cyber (Center for AI Safety)

Nesta fase realizamos a avaliação oficial do CyberRott v1.2 no benchmark **WMDP-Cyber (1.987 questões)**:
1. Avalia a acurácia generativa com cadeia de pensamento (<think>...</think>) e extração de alternativa.
2. Utiliza o motor **vLLM** em bfloat16 para altíssimo throughput (conclusão em poucos minutos na GPU).
3. Salva os relatórios detalhados em eval_results/ para inclusão oficial no Model Card do Hugging Face.

In [ ]:
# Avaliação 100% Pública via Hugging Face Hub (qualquer usuário pode rodar sem Google Drive)
!python scripts/eval_wmdp_vllm.py \n    --model_name johnknevescarrijo/CyberRott-v1.2-GRPO \n    --max_tokens 512 \n    --output_dir eval_results
